<a id="bresenham"></a>

$$
\text{布雷森汉姆画线算法 (Bresenham's Line Algorithm): 只用整数加减法画直线} \\ \quad \\
\text{先考虑第一象限且斜率 } 0 \leq m \leq 1 \text{ 的情况 (其他情况交换 x, y 或改变步进方向): } dx = x_1 - x_0 \geq dy = y_1 - y_0 \geq 0 \\
\text{x 每次加 1, 每列只画一个像素; 下一个像素只有两个候选: } (x + 1, y) \text{ 或 } (x + 1, y + 1) \\ \quad \\
\text{1. 直线的隐式方程, 两边乘以 dx 去掉分母 (斜率 } m = dy / dx \text{):} \\
f(x, y) = dy \, (x - x_0) - dx \, (y - y_0) \quad (\text{直线上 } f = 0 \text{, 直线下方 } f > 0 \text{, 上方 } f < 0) \\
\text{2. 判断两个候选像素的中点 } (x + 1, y + \frac{1}{2}) \text{ 在直线的哪一边:} \\
f(x + 1, y + \tfrac{1}{2}) \geq 0 \implies \text{中点在直线下方 (或线上), 直线更靠近 } y + 1 \text{, 选 } (x + 1, y + 1) \\
f(x + 1, y + \tfrac{1}{2}) < 0 \implies \text{选 } (x + 1, y) \\
\text{3. 乘以 2 去掉 } \frac{1}{2} \text{, 定义决策变量 (Decision Variable) } D = 2 f(x + 1, y + \tfrac{1}{2}) \\
\text{起点 } (x_0, y_0) \text{: } D_0 = 2 \left( dy - \frac{dx}{2} \right) = 2 dy - dx \\
\text{4. 增量更新 (不用每次重新算 f): } f \text{ 是线性的, x 加 1 时 f 加 dy, y 加 1 时 f 减 dx} \\
\text{选 } (x + 1, y) \text{: 下一个中点 x 加 1} \implies D \mathrel{+}= 2 dy \\
\text{选 } (x + 1, y + 1) \text{: 下一个中点 x 加 1, y 加 1} \implies D \mathrel{+}= 2 dy - 2 dx \\
\text{全程只有整数加减法和比较, 没有乘除法和浮点数 (早期硬件上非常重要)}
$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(suppress=True, precision=4)

def plot_nicely():
    for ax in plt.gcf().get_axes():
        ax.grid(True, which="both", alpha=0.3)
        if ax.get_legend_handles_labels()[0]:
            ax.legend(fontsize='small')
    plt.tight_layout()

def bresenham(x0, y0, x1, y1):
    steep = abs(y1 - y0) > abs(x1 - x0)
    if steep:                                  # 斜率 > 1: 交换 x, y, 让 y 当作主方向
        x0, y0, x1, y1 = y0, x0, y1, x1
    if x0 > x1:                                # 从左往右画
        x0, y0, x1, y1 = x1, y1, x0, y0
    dx, dy = x1 - x0, abs(y1 - y0)
    y_step = 1 if y0 < y1 else -1
    D = 2 * dy - dx
    y = y0
    pixels = []
    for x in range(x0, x1 + 1):
        pixels.append((y, x) if steep else (x, y))
        if D >= 0:
            y += y_step
            D -= 2 * dx
        D += 2 * dy
    return np.array(pixels)

# 1. 第一象限斜率 <= 1: 和 "理想直线的 y 四舍五入" 完全一致
x0, y0, x1, y1 = 2, 3, 23, 11
pix = bresenham(x0, y0, x1, y1)
ideal_y = y0 + (y1 - y0) / (x1 - x0) * (pix[:, 0] - x0)
print(f'Bresenham == round(理想 y) ? {np.array_equal(pix[:, 1], np.floor(ideal_y + 0.5).astype(int))}')
print(f'输出全是整数 ? {pix.dtype.kind == "i"}')

# 2. 随机方向的直线: 像素数 = max(|dx|, |dy|) + 1, 次方向上离理想直线不超过 0.5
rng = np.random.default_rng(0)
max_dev = 0
for _ in range(2000):
    a, b = rng.integers(-50, 50, 2), rng.integers(-50, 50, 2)
    if np.array_equal(a, b):
        continue
    px = bresenham(*a, *b)
    dx, dy = b - a
    assert len(px) == max(abs(dx), abs(dy)) + 1
    if abs(dx) >= abs(dy):
        dev = np.abs(px[:, 1] - (a[1] + dy / dx * (px[:, 0] - a[0])))
    else:
        dev = np.abs(px[:, 0] - (a[0] + dx / dy * (px[:, 1] - a[1])))
    max_dev = max(max_dev, dev.max())
print(f'2000 条随机直线: 次方向上离理想直线的最大距离 = {max_dev:.4f} (<= 0.5)')

plt.figure(figsize=(4, 3))
for (px_, py_) in pix:
    plt.gca().add_patch(plt.Rectangle((px_ - 0.5, py_ - 0.5), 1, 1, color='C0', alpha=0.5))
plt.plot([x0, x1], [y0, y1], 'r', label='ideal line')
plt.plot(pix[:, 0], pix[:, 1], 'k.', label='Bresenham pixels')
plt.gca().set_aspect('equal')
plot_nicely()
plt.show()

<a id="edge-function"></a>

$$
\text{三角形光栅化 (Triangle Rasterization): 边函数 (Edge Function)} \\
\text{2D 叉积 (z 分量): } (b - a) \times (p - a) = (b_x - a_x)(p_y - a_y) - (b_y - a_y)(p_x - a_x) \\
E_{ab}(p) = \det \begin{bmatrix} b_x - a_x & p_x - a_x \\ b_y - a_y & p_y - a_y \end{bmatrix} = 2 \times \text{三角形 (a, b, p) 的有向面积 (Signed Area)} \\ \quad \\
\text{符号的意义: } E_{ab}(p) > 0 \text{, p 在有向边 } a \to b \text{ 的左侧; < 0 在右侧; = 0 在直线上} \\
\text{一个点在三角形内部} \iff \text{它在三条边 } a \to b, \ b \to c, \ c \to a \text{ 的同一侧} \\
\iff E_{bc}(p), \ E_{ca}(p), \ E_{ab}(p) \text{ 和 } E_{ab}(c) \text{ (整个三角形的有向面积 × 2) 同号} \\ \quad \\
\text{E 是 p 的线性函数 (仿射函数): 往右移动一个像素, E 增加一个常数 } -(b_y - a_y) \text{, 所以 GPU 可以并行地对每个像素算}
$$

<a id="barycentric"></a>

$$
\text{重心坐标 (Barycentric Coordinates)} \\
\text{三角形内任意一点可以写成三个顶点的加权平均: } p = \lambda_0 a + \lambda_1 b + \lambda_2 c, \quad \lambda_0 + \lambda_1 + \lambda_2 = 1 \\ \quad \\
\text{推导: 把 } \lambda_0 = 1 - \lambda_1 - \lambda_2 \text{ 代入: } p - a = \lambda_1 (b - a) + \lambda_2 (c - a) \\
\begin{bmatrix} b - a & c - a \end{bmatrix} \begin{bmatrix} \lambda_1 \\ \lambda_2 \end{bmatrix} = p - a \quad (2 \times 2 \text{ 线性方程组}) \\
\text{克莱姆法则 (Cramer's Rule): } \lambda_1 = \frac{\det[p - a, \ c - a]}{\det[b - a, \ c - a]}, \quad \lambda_2 = \frac{\det[b - a, \ p - a]}{\det[b - a, \ c - a]} \\
\text{分母 = 2 × 三角形 abc 的面积, 分子 = 2 × 子三角形的面积:} \\
\lambda_0 = \frac{\text{Area}(p, b, c)}{\text{Area}(a, b, c)} = \frac{E_{bc}(p)}{E_{ab}(c)}, \quad \lambda_1 = \frac{\text{Area}(a, p, c)}{\text{Area}(a, b, c)} = \frac{E_{ca}(p)}{E_{ab}(c)}, \quad \lambda_2 = \frac{\text{Area}(a, b, p)}{\text{Area}(a, b, c)} = \frac{E_{ab}(p)}{E_{ab}(c)} \\
\text{(p 越靠近顶点 a, 对面的子三角形 pbc 越大, a 的权重越大)} \\ \quad \\
\text{用途: 顶点上的任何属性 (颜色, 法向量, 纹理坐标, 深度) 都用 } \lambda \text{ 插值到三角形内部: } u(p) = \lambda_0 u_a + \lambda_1 u_b + \lambda_2 u_c \\
\text{三个 } \lambda \text{ 都 } \geq 0 \text{ 就是在三角形内, 和边函数的判断是同一件事}
$$

数学基础: [行列式 (面积)](../math_basics/linear_algebra.ipynb#determinant) · [解线性方程组 / 矩阵的逆](../math_basics/linear_algebra.ipynb#inverse)

In [ ]:
def edge(a, b, px, py):
    return (b[0] - a[0]) * (py - a[1]) - (b[1] - a[1]) * (px - a[0])

def rasterize_triangle(v2d, W, H):
    # v2d: 3 个屏幕坐标顶点. 在每个像素中心算边函数, 返回 (是否在内部, 重心坐标)
    px, py = np.meshgrid(np.arange(W) + 0.5, np.arange(H) + 0.5)
    a, b, c = v2d
    area2 = edge(a, b, c[0], c[1])
    lam = np.stack([edge(b, c, px, py), edge(c, a, px, py), edge(a, b, px, py)]) / area2
    inside = np.all(lam >= 0, axis=0)
    return inside, lam

W, H = 160, 120
tri = np.array([[20.0, 15.0], [140.0, 40.0], [60.0, 110.0]])
inside, lam = rasterize_triangle(tri, W, H)
print(f'三个重心坐标的和 == 1 ? {np.allclose(lam.sum(axis=0), 1)}')

# 用面积比算的 lambda == 解线性方程组的 lambda
p_test = np.array([70.3, 52.8])
M2 = np.column_stack([tri[1] - tri[0], tri[2] - tri[0]])
l12 = np.linalg.solve(M2, p_test - tri[0])
_, lam_p = rasterize_triangle(tri, 1, 1)
area = lambda A, B, Cc: 0.5 * edge(A, B, Cc[0], Cc[1])
lam_area = np.array([area(p_test, tri[1], tri[2]), area(tri[0], p_test, tri[2]), area(tri[0], tri[1], p_test)]) / area(*tri)
print(f'面积比 lambda = {lam_area}, 解方程组 lambda = {np.concatenate([[1 - l12.sum()], l12])}')
print(f'重心坐标还原出 p = {lam_area @ tri}, 原来的 p = {p_test}')

# 覆盖的像素数 ~ 三角形面积
print(f'覆盖像素数 = {inside.sum()}, 三角形面积 = {abs(area(*tri)):.1f}')

# 用重心坐标插值颜色 (三个顶点分别是红, 绿, 蓝)
colors = np.eye(3)
img = np.ones((H, W, 3))
img[inside] = np.einsum('kn,kc->nc', lam[:, inside], colors)
plt.figure(figsize=(4, 3))
plt.imshow(img)
plt.plot(*np.vstack([tri, tri[:1]]).T, 'k', linewidth=0.8, label='triangle')
plot_nicely()
plt.show()

<a id="perspective-correct"></a>

$$
\text{透视校正插值 (Perspective-Correct Interpolation)} \\
\text{问题: 屏幕上的重心坐标 } \lambda_i \text{ 不等于 3D 空间里的重心坐标 } \mu_i \text{, 因为透视除法不是线性的} \\ \quad \\
\text{推导: 3D 三角形上的点 } P = \sum \mu_i P_i \text{, 属性 } u = \sum \mu_i u_i \text{ (属性在 3D 中线性变化)} \\
\text{投影矩阵是线性的: 裁剪坐标 } (x_c, w_c) = \sum \mu_i (x_{c,i}, w_i) \\
\text{屏幕坐标 } x_s = \frac{x_c}{w_c} = \frac{\sum \mu_i x_{c,i}}{\sum \mu_j w_j} = \sum_i \underbrace{\frac{\mu_i w_i}{\sum_j \mu_j w_j}}_{\lambda_i} \cdot \frac{x_{c,i}}{w_i} \\
\text{而屏幕上的重心坐标满足 } x_s = \sum \lambda_i x_{s,i} = \sum \lambda_i \frac{x_{c,i}}{w_i} \text{, 对比得 } \lambda_i = \frac{\mu_i w_i}{\sum_j \mu_j w_j} \\
\text{反解 } \mu \text{: } \mu_i \propto \frac{\lambda_i}{w_i} \text{, 再归一化使和为 1} \implies \mu_i = \frac{\lambda_i / w_i}{\sum_j \lambda_j / w_j} \\ \quad \\
u = \sum \mu_i u_i = \frac{\sum_i \lambda_i \, (u_i / w_i)}{\sum_i \lambda_i \, (1 / w_i)} \\
\text{结论: 在屏幕空间里线性插值的应该是 } \frac{u}{w} \text{ 和 } \frac{1}{w} \text{, 再相除; 直接线性插值 u (仿射插值) 会让纹理扭曲} \\ \quad \\
\text{推论: } z_{ndc} = \frac{A z + B}{-z} = -A + \frac{B}{w} \text{ (} w = -z \text{) 是 } \frac{1}{w} \text{ 的线性函数, 所以深度可以直接在屏幕空间线性插值}
$$

数学基础: [透视投影矩阵的推导](perspective_projection.ipynb#perspective-derivation)

In [ ]:
fov, n_near, f_far = 60.0, 0.1, 100.0
aspect = W / H
F = 1 / np.tan(np.deg2rad(fov) / 2)
P_mat = np.array([[F / aspect, 0, 0, 0],
                  [0, F, 0, 0],
                  [0, 0, -(f_far + n_near) / (f_far - n_near), -2 * f_far * n_near / (f_far - n_near)],
                  [0, 0, -1, 0]])

def project(v_cam):
    clip = np.column_stack([v_cam, np.ones(len(v_cam))]) @ P_mat.T
    w = clip[:, 3]
    ndc = clip[:, :3] / w[:, None]
    screen = np.column_stack([(ndc[:, 0] + 1) / 2 * W, (1 - ndc[:, 1]) / 2 * H])
    return screen, ndc[:, 2], w

# 地面 y = -1 上的一个长方形 (相机坐标), 拆成两个三角形, 每个顶点带纹理坐标 (u, v)
quad = np.array([[-1.0, -1, -2], [1.0, -1, -2], [1.0, -1, -8], [-1.0, -1, -8]])
uv = np.array([[0.0, 0], [1.0, 0], [1.0, 1], [0.0, 1]])
tris = [(0, 1, 2), (0, 2, 3)]
screen, z_ndc_v, w_v = project(quad)

def checker(u, v, k=8):
    return ((np.floor(u * k) + np.floor(v * k)) % 2).astype(float)

uv_affine = np.full((H, W, 2), np.nan)
uv_correct = np.full((H, W, 2), np.nan)
for tri_idx in tris:
    inside, lam = rasterize_triangle(screen[list(tri_idx)], W, H)
    L = lam[:, inside]                               # (3, 像素数)
    uv_t, w_t = uv[list(tri_idx)], w_v[list(tri_idx)]
    uv_affine[inside] = (L.T @ uv_t)
    uv_correct[inside] = ((L / w_t[:, None]).T @ uv_t) / (L / w_t[:, None]).sum(axis=0)[:, None]

# 独立验证: 对每个像素发射一条光线, 和地面 y = -1 求交点, 直接算真实的纹理坐标
covered = ~np.isnan(uv_correct[..., 0])
py_, px_ = np.nonzero(covered)
x_ndc = 2 * (px_ + 0.5) / W - 1
y_ndc = 1 - 2 * (py_ + 0.5) / H
d = np.column_stack([x_ndc / F * aspect, y_ndc / F, -np.ones_like(x_ndc)])
t_hit = -1 / d[:, 1]
hit = d * t_hit[:, None]
uv_exact = np.column_stack([(hit[:, 0] + 1) / 2, (-hit[:, 2] - 2) / 6])
print(f'透视校正插值 == 光线求交的真实纹理坐标 ? {np.allclose(uv_correct[covered], uv_exact)}')
print(f'仿射插值的最大误差 = {np.max(np.abs(uv_affine[covered] - uv_exact)):.4f}')

fig, axs = plt.subplots(1, 2, figsize=(8, 3))
for ax, uvs, title in [(axs[0], uv_affine, 'affine (wrong)'), (axs[1], uv_correct, 'perspective-correct')]:
    img = np.ones((H, W))
    img[covered] = checker(uvs[covered, 0], uvs[covered, 1])
    ax.imshow(img, cmap='gray')
    ax.plot(*np.vstack([screen[[0, 2]]]).T, 'r', linewidth=0.8, label='triangle seam')
    ax.set_title(title)
plot_nicely()
plt.show()

<a id="z-buffer"></a>

$$
\text{深度缓冲 (Z-Buffer / Depth Buffer): 解决遮挡} \\
\text{每个像素保存目前为止最近的深度, 新片元 (fragment) 的深度更小才覆盖颜色} \\
\text{for 每个三角形, for 每个覆盖的像素 p: } \quad \text{if } z(p) < \text{zbuf}[p] \text{: zbuf}[p] = z(p), \ \text{color}[p] = \text{color} \\ \quad \\
\text{深度 } z_{ndc} \text{ 用屏幕重心坐标直接线性插值 (上面推导过它是 } \frac{1}{w} \text{ 的线性函数, 在屏幕空间是线性的)} \\ \quad \\
\text{对比画家算法 (Painter's Algorithm): 按三角形的平均深度从远到近排序再画} \\
\text{两个三角形互相穿插时, 没有一个排序是对的 (一部分 A 在 B 前面, 另一部分 B 在 A 前面), 而 z-buffer 是逐像素比较, 没有这个问题}
$$

In [ ]:
# 两个互相穿插的三角形 (相机坐标)
tri_A = np.array([[-1.5, -1.0, -4.0], [1.5, -1.0, -4.0], [0.0, 1.5, -7.0]])    # 往远处倾斜
tri_B = np.array([[-1.5, 0.8, -7.0], [1.5, 0.8, -7.0], [0.0, -1.2, -3.5]])     # 往近处倾斜
colors_AB = [np.array([0.9, 0.3, 0.2]), np.array([0.2, 0.4, 0.9])]

zbuf = np.full((H, W), np.inf)
img_z = np.ones((H, W, 3))
coverage = []
for tri_cam, col in zip([tri_A, tri_B], colors_AB):
    s, zn, _ = project(tri_cam)
    inside, lam = rasterize_triangle(s, W, H)
    z_pix = np.einsum('k,khw->hw', zn, lam)
    closer = inside & (z_pix < zbuf)
    zbuf[closer] = z_pix[closer]
    img_z[closer] = col
    coverage.append(inside)

# 画家算法: 按顶点平均深度从远到近画
img_p = np.ones((H, W, 3))
order = np.argsort([tri_A[:, 2].mean(), tri_B[:, 2].mean()])     # z 越小越远, 先画
for idx in order:
    s, _, _ = project([tri_A, tri_B][idx])
    inside, _ = rasterize_triangle(s, W, H)
    img_p[inside] = colors_AB[idx]

# 独立验证: 对每个两者都覆盖的像素, 光线和两个三角形所在平面求交, 距离近的那个应该可见
both = coverage[0] & coverage[1]
py_, px_ = np.nonzero(both)
d = np.column_stack([(2 * (px_ + 0.5) / W - 1) / F * aspect, (1 - 2 * (py_ + 0.5) / H) / F, -np.ones(len(px_))])
def ray_plane_t(tri_cam, d):
    nrm = np.cross(tri_cam[1] - tri_cam[0], tri_cam[2] - tri_cam[0])
    return (nrm @ tri_cam[0]) / (d @ nrm)          # 光线 o + t d (o = 0) 和平面 n.(x - p0) = 0
A_front = ray_plane_t(tri_A, d) < ray_plane_t(tri_B, d)
zbuf_says_A = np.all(img_z[both] == colors_AB[0], axis=1)
painter_says_A = np.all(img_p[both] == colors_AB[0], axis=1)
print(f'两个三角形都覆盖的像素数 = {both.sum()}')
print(f'z-buffer 和光线求交一致的比例 = {np.mean(zbuf_says_A == A_front):.4f}')
print(f'画家算法和光线求交一致的比例 = {np.mean(painter_says_A == A_front):.4f}')

fig, axs = plt.subplots(1, 2, figsize=(8, 3))
axs[0].imshow(img_z)
axs[0].set_title('z-buffer')
axs[1].imshow(img_p)
axs[1].set_title("painter's algorithm")
plot_nicely()
plt.show()